### 1. Setup and Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd

### 2. File Setup

In [2]:
database = 'DB1'
participant = 'P01'

data_folder = f'../data/{database}/{participant}'
files = [f for f in os.listdir(data_folder) if f.endswith('.trc')]  # Get list of all .trc files

output_folder = f'../data/{database}_retargeted/{participant}'
os.makedirs(output_folder, exist_ok=True) # Create the output directory if it doesn't exist

translation_file = '../data/translation_table.xlsx'

print(f"Found {len(files)} C3D files to process.")

Found 1 C3D files to process.


## 3. Analyzing the labels of the translation_table
Reading the `translation_table` containing our marker mappings.
* Column 1 contains the Model (Reference) Labels.
* Columns 2 to 5 contain the different Databases Labels.

In [7]:
df_translation = pd.read_excel(translation_file) # Read the excel file

# Extract column names based on their order
ref_name = df_translation.columns[0]
db_names = df_translation.columns[1:12]

db_mappings = {db: {} for db in db_names}  # Dictionary format: { 'DB_Name': { 'old_label': 'Ref_Label' } }

for index, row in df_translation.iterrows():
    ref_label = row[ref_name] # Grabs reference label    
    if pd.isna(ref_label):   # Skip if there's no reference label for this row
        continue

    for i, db_col in enumerate(db_names):
        old_label = row[db_col]
        if pd.notna(old_label):   # If the database has a label for this row
            clean_old = str(old_label).strip()
            db_mappings[db_col][clean_old] = str(ref_label).strip()  # Assign the reference label to the specific database's dictionary

print(f"Loaded {df_translation[ref_name].dropna().count()} total reference markers.")

Loaded 42 total reference markers.


## 4. Relabeling (aka retargeting)

In [8]:
processed_data = []  # List to temporarily store our modified c3d objects before saving

for file_name in files:
    file_path = os.path.join(data_folder, file_name)

    # TRC files are plain text, so we read the lines directly
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # In TRC format, line 4 (index 3) contains the marker labels, tab-separated
    header_elements = lines[3].strip('\n').split('\t')

    # Extract only the actual marker names (ignore empty tabs and 'Frame#', 'Time')
    clean_current_labels = [lbl.strip() for lbl in header_elements if lbl.strip() and lbl.strip() not in ['Frame#', 'Time']]

    # 1. SCORING PHASE: Identify which database this file belongs to
    db_hits = {db: 0 for db in db_names}
    for label in clean_current_labels:
        for db_name, mapping in db_mappings.items():
            if label in mapping:
                db_hits[db_name] += 1

    primary_db = max(db_hits, key=db_hits.get)  # Find the database with the most hits
    max_hits = db_hits[primary_db]

    # 2. RELABELING PHASE
    relabeled_count = 0
    not_relabeled_count = 0
    relabeled_names = []

    if max_hits > 0:
        winning_mapping = db_mappings[primary_db]
        new_header_elements = []

        for element in header_elements:
            clean_label = element.strip()

            if clean_label in winning_mapping and winning_mapping[clean_label] != clean_label:
                new_header_elements.append(winning_mapping[clean_label])
                relabeled_names.append(f"'{clean_label}' to '{winning_mapping[clean_label]}'")
                relabeled_count += 1
            else:
                new_header_elements.append(element)  # Keep original label or empty tab
                if clean_label and clean_label not in ['Frame#', 'Time']:
                    not_relabeled_count += 1

        # Overwrite the 4th line in our text array with the new tab-separated labels
        lines[3] = '\t'.join(new_header_elements) + '\n'

        if relabeled_count > 0: # Only save to memory if we actually changed something
            processed_data.append((file_name, lines)) 
        print(f"File '{file_name}' ({primary_db}) -> {relabeled_count} markers were relabeled, while {not_relabeled_count} markers were not.")
        #if relabeled_count > 0:
        #    print(f"    Changes: {', '.join(relabeled_names)}\n")
    else:
        print(f"File '{file_name}' -> 0 markers were relabeled, while {len(clean_current_labels)} markers were not.")

File 'LG_Walk_Slower03_filtered.trc' (DB Dimitrov) -> 14 markers were relabeled, while 21 markers were not.


## 5. Exporting data after retarget


In [9]:
print("\nStarting TRC export...")
for file_name, trc_lines in processed_data:
    new_file_name = file_name.replace('.trc', '_relabeled.trc')  # Rename the file 
    output_path = os.path.join(output_folder, new_file_name)

    # Write the modified lines back to a text file
    with open(output_path, 'w') as f:
        f.writelines(trc_lines)

    print(f"Exported: {new_file_name}")

print("\nProcess complete! All TRC files saved to the output folder.")


Starting TRC export...
Exported: LG_Walk_Slower03_filtered_relabeled.trc

Process complete! All TRC files saved to the output folder.
